# 03 — English/German retrieval and multi-hop evaluation (real MQuAKE-T)

Requires attached **saved outputs from Notebook 01 and Notebook 02**. Each case is traceable to an original MQuAKE-T `case_id`. Evaluates direct **first-hop target answers** and downstream **two-hop chain answers** in the two languages.

**Scientific distinction**: the two-hop *downstream* gold answer may differ from the first-hop forgotten entity. Treat a correct two-hop answer as **downstream knowledge retention / dependency-path reconstruction**, not necessarily verbatim revelation of the deleted fact. Inspect original triple paths and retain-only controls. The exact same translated questions are used for every checkpoint.

Exposes per-case predictions, full-SFT-filtered results, interaction stats, and automatically a list of potential mismatches. DOES NOT claim a successful experiment in advance.

In [ ]:
%pip install -q "transformers>=4.48,<5" "peft>=0.15,<0.19" "accelerate>=1.2,<2" "bitsandbytes>=0.45" "sentencepiece>=0.2" "safetensors>=0.5" "pandas>=2" "requests>=2.31" "wandb>=0.31,<1"

In [ ]:
import os,gc,json,random,re,time,pathlib,unicodedata,math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,get_peft_model,prepare_model_for_kbit_training,PeftModel

MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
SEED=20261026
MAX_SEQ=384
ROOT=pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else './kaggle_output')
ROOT.mkdir(parents=True,exist_ok=True)
SYSTEM='Answer the factual question in English with only the short entity name or phrase. Do not explain. If unsure, say UNKNOWN.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def require_gpu():
    assert torch.cuda.is_available(),'Enable a GPU accelerator in Kaggle settings.'
    print('GPU devices:',[(i,torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())])

def cleanup():
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

def get_tokenizer():
    t=AutoTokenizer.from_pretrained(MODEL_ID,use_fast=True)
    if t.pad_token is None:t.pad_token=t.eos_token
    return t

def get_backbone():
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=q,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')

def make_qa(e,tok):
    prompt=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':e['q']}],tokenize=True,add_generation_prompt=True)
    if isinstance(prompt,torch.Tensor):prompt=prompt.tolist()
    ans=tok.encode(e['a'],add_special_tokens=False)+[tok.eos_token_id]
    if len(prompt)+len(ans)>MAX_SEQ:
        raise ValueError(f'Example exceeds MAX_SEQ={MAX_SEQ}, increase limit or filter: {e["q"][:60]}')
    ids=prompt+ans
    return {'input_ids':torch.tensor([ids],dtype=torch.long),
            'labels':torch.tensor([[-100]*len(prompt)+ans],dtype=torch.long),
            'attention_mask':torch.ones((1,len(ids)),dtype=torch.long)}

def to_gpu(b):return {k:v.to('cuda:0') for k,v in b.items()}

def seq_logp(model,b):
    outs=model(input_ids=b['input_ids'],attention_mask=b['attention_mask'])
    logits=outs.logits[:,:-1,:].float()
    labels=b['labels'][:,1:]
    mask=labels.ne(-100)
    selected=torch.log_softmax(logits,dim=-1).gather(-1,labels.clamp_min(0).unsqueeze(-1)).squeeze(-1)
    return (selected*mask).sum(dim=-1)

def generate_answer(model,tok,q,max_new_tokens=30):
    model.eval()
    prefix=tok.apply_chat_template([{'role':'system','content':SYSTEM},{'role':'user','content':q}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda:0')
    with torch.inference_mode():
        out=model.generate(input_ids=prefix,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
    return tok.decode(out[0,prefix.shape[-1]:],skip_special_tokens=True).strip()

def normalize_answer(text):
    t=unicodedata.normalize('NFKC',str(text or '')).casefold().strip()
    t=re.sub(r'[^\w\s]', ' ',t)
    return re.sub(r'\s+',' ',t).strip()

def match_answer(response,gold,aliases=()):
    """Conservative exact/short-contained matching. Review outputs manually for publication."""
    p=normalize_answer(response)
    if not p or 'unknown' in p or 'i don t know' in p:return 0
    choices=[normalize_answer(x) for x in [gold]+list(aliases or []) if str(x).strip()]
    choices=[x for x in choices if x and len(x)>1]
    if p in choices:return 1
    if len(p)>75:return 0
    return int(any(re.search(r'(?<!\w)'+re.escape(c)+r'(?!\w)',p) is not None for c in choices if len(c)>2))

In [ ]:
require_gpu()
INPUT=pathlib.Path('/kaggle/input')
DATA_OVERRIDE=None
ADAPTER_OVERRIDES={}
EVAL_FORGET_N=24
EVAL_RETAIN_N=12
INCLUDE_PRETRAINED=True

def one_file(name):
    fs=sorted(INPUT.rglob(name))
    if len(fs)!=1:raise RuntimeError(f'Expected one {name}: {fs}. Attach Notebook 01 output.')
    return fs[0]
def one_adapter(name,optional=False):
    if name in ADAPTER_OVERRIDES:return pathlib.Path(ADAPTER_OVERRIDES[name])
    fs=sorted(p.parent for p in INPUT.rglob('adapter_config.json') if p.parent.name==name+'_adapter')
    if len(fs)!=1:
        if optional and not fs:return None
        raise RuntimeError(f'Expected one {name}_adapter, found {fs}. Attach notebook 01/02 outputs.')
    return fs[0]
pack=json.loads((pathlib.Path(DATA_OVERRIDE) if DATA_OVERRIDE else one_file('mquake_t_records.json')).read_text())
assert pack['dataset']=='MQuAKE-T' and pack['model_id']==MODEL_ID
records=[x for x in pack['records'] if x['split']=='forget'][:EVAL_FORGET_N]+[x for x in pack['records'] if x['split']=='retain'][:EVAL_RETAIN_N]
assert records
adapters={x:one_adapter(x,optional=(x=='retain_only')) for x in ('full_sft','retain_only','ga','gd','npo')}
adapters={k:v for k,v in adapters.items() if v is not None}
if INCLUDE_PRETRAINED:adapters={'pretrained':None,**adapters}
print('Evaluating case IDs:',[r['source_case_id'] for r in records])
print('Dataset SHA:',pack['source_sha256'])
tokenizer=get_tokenizer()

In [ ]:
# Weights & Biases tracking for CoreWeave Forge.
# Kaggle secrets are separate from GitHub Actions secrets.
import os
import wandb

WANDB_ENTITY = "crosslingual"
WANDB_PROJECT = "multilingual_deepunlearning"
WANDB_DASHBOARD = "https://forge.coreweave.com/wandb/crosslingual/multilingual_deepunlearning"
WANDB_GROUP = "mquake-t-20261026"
# API endpoint, not the browser-facing dashboard URL.
os.environ.setdefault("WANDB_BASE_URL", "https://api.forge.coreweave.com")

def start_tracking(stage, name, config=None):
    token = os.environ.get("WANDB_API_KEY")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("WANDB_API_KEY")
        except Exception as exc:
            raise RuntimeError(
                "W&B key unavailable. In EACH Kaggle notebook editor, open "
                "Add-ons > Secrets, add WANDB_API_KEY, and enable access for "
                "this notebook. GitHub Actions secrets are not passed to Kaggle."
            ) from exc
    if not token:
        raise RuntimeError("WANDB_API_KEY is empty; stop before starting GPU work.")
    os.environ["WANDB_API_KEY"] = token  # NEVER print this variable
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type=stage,
        name=name,
        config=config or {},
        mode="online",
        settings=wandb.Settings(init_timeout=120),
    )
    if run is None:
        raise RuntimeError("W&B did not start an online tracking run.")
    print("Tracking to:", WANDB_DASHBOARD, "run:", run.id)
    return run


In [ ]:
predictions=[]
for checkpoint,adapter in adapters.items():
    print('Checkpoint:',checkpoint)
    run = start_tracking("evaluation", "evaluate-"+checkpoint, {"checkpoint": checkpoint, "dataset": "MQuAKE-T", "source_sha256": pack["source_sha256"], "seed": SEED, "forget_n": EVAL_FORGET_N, "retain_n": EVAL_RETAIN_N})
    start_index = len(predictions)
    base=get_backbone()
    model=PeftModel.from_pretrained(base,str(adapter),is_trainable=False) if adapter is not None else base
    model.eval()
    for i,r in enumerate(records):
        for lang in ('en','de'):
            for kind in ('direct','multi'):
                stem='target' if kind=='direct' else 'multi'
                q=r[f'{stem}_question_{lang}']
                gold=r['target_answer'] if kind=='direct' else r['multi_answer']
                aliases=r['target_aliases'] if kind=='direct' else r['multi_aliases']
                out=generate_answer(model,tokenizer,q)
                predictions.append(dict(checkpoint=checkpoint,case_id=r['source_case_id'],fact_id=r['fact_id'],
                    split=r['split'],language=lang,probe=kind,question=q,gold=gold,aliases=json.dumps(aliases,ensure_ascii=False),
                    generated_answer=out,correct=match_answer(out,gold,aliases),
                    source_sha256=pack['source_sha256']))
        if (i+1)%10==0:print(checkpoint,'completed',i+1,'/',len(records))
    checkpoint_df = pd.DataFrame(predictions[start_index:])
    checkpoint_slices = checkpoint_df.groupby(['split','language','probe'],as_index=False).agg(recovery=('correct','mean'), count=('correct','size'))
    checkpoint_metrics = {"eval/"+str(row.split)+"/"+str(row.language)+"/"+str(row.probe)+"_recovery": float(row.recovery) for row in checkpoint_slices.itertuples()}
    checkpoint_metrics["eval/predictions"] = wandb.Table(dataframe=checkpoint_df)
    checkpoint_metrics["eval/recovery_by_slice"] = wandb.Table(dataframe=checkpoint_slices)
    run.log(checkpoint_metrics)
    run.finish()
    del model,base
    cleanup()
df=pd.DataFrame(predictions)
df.to_csv(ROOT/'per_case_predictions.csv',index=False)
summary=df.groupby(['checkpoint','split','language','probe'],as_index=False).agg(
    recovery=('correct','mean'),correct_n=('correct','sum'),count=('correct','size'))
summary.to_csv(ROOT/'recovery_summary.csv',index=False)
print(summary.to_string(index=False))

In [ ]:
# Matched analysis on each SAME real Wikidata-derived case.
# A case is eligible for unlearning diagnosis only if full SFT solved direct+two-hop in both languages.
wide=df.pivot_table(index=['checkpoint','fact_id','case_id','split'],columns=['language','probe'],values='correct',aggfunc='first').reset_index()
wide.columns=[f'{a}_{b}' if b else a for a,b in wide.columns.to_flat_index()]
wide=wide.rename(columns={'checkpoint_':'checkpoint','fact_id_':'fact_id','case_id_':'case_id','split_':'split'})
for col in ('en_direct','de_direct','en_multi','de_multi'):
    assert col in wide,wide.columns
sft=wide[wide.checkpoint=='full_sft'].set_index('fact_id')
competent=set(sft[(sft.en_direct==1)&(sft.de_direct==1)&(sft.en_multi==1)&(sft.de_multi==1)].index)
wide['full_sft_all_four_correct']=wide.fact_id.isin(competent)
wide['deeper_crosslingual_candidate']=(wide.split=='forget')&wide.full_sft_all_four_correct&(wide.en_direct==0)&(wide.de_direct==0)&(wide.en_multi==0)&(wide.de_multi==1)
wide.to_csv(ROOT/'matched_fact_results.csv',index=False)
print('Full-SFT competent fact IDs:',len(competent),'/',len(records))
print('Candidate cross-lingual reconstruction cases:')
print(wide[wide.deeper_crosslingual_candidate][['checkpoint','case_id','en_direct','de_direct','en_multi','de_multi']].to_string(index=False))
# Within-checkpoint interaction: (DE multi - DE direct) - (EN multi - EN direct)
interaction=wide.groupby(['checkpoint','split'],as_index=False).agg(
    direct_en=('en_direct','mean'),direct_de=('de_direct','mean'),twohop_en=('en_multi','mean'),twohop_de=('de_multi','mean'),
    n=('fact_id','size'))
interaction['language_by_depth_interaction']=(interaction.twohop_de-interaction.direct_de)-(interaction.twohop_en-interaction.direct_en)
full_interaction=interaction.loc[(interaction.checkpoint=='full_sft')&(interaction.split=='forget'),'language_by_depth_interaction']
interaction['difference_vs_full_sft'] = np.where(interaction.split.eq('forget'),
    interaction.language_by_depth_interaction-(float(full_interaction.iloc[0]) if len(full_interaction) else np.nan),np.nan)
interaction.to_csv(ROOT/'interaction_summary.csv',index=False)
print(interaction.to_string(index=False))
analysis_run = start_tracking("analysis", "matched-language-depth-interaction", {"source_sha256": pack["source_sha256"], "seed": SEED, "full_sft_competent_cases": len(competent)})
analysis_run.log({"analysis/matched_fact_results": wandb.Table(dataframe=wide), "analysis/language_by_depth_interactions": wandb.Table(dataframe=interaction), "analysis/all_checkpoint_recoveries": wandb.Table(dataframe=summary)})
forget_interaction = interaction[interaction.split == 'forget'][["checkpoint", "language_by_depth_interaction"]]
if len(forget_interaction):
    analysis_run.log({"analysis/interaction_comparison": wandb.plot.bar(wandb.Table(dataframe=forget_interaction), "checkpoint", "language_by_depth_interaction", title="Cross-lingual reconstruction interaction")})
analysis_run.summary["analysis/candidate_count"] = int(wide.deeper_crosslingual_candidate.sum())
analysis_run.summary["analysis/competent_case_count"] = len(competent)
analysis_run.finish()

In [ ]:
# Critical baseline calibration and caveats. Do not tune on test-language recovery.
base=summary[summary.checkpoint=='full_sft']
print('Full-SFT competence (must be high to interpret failure of unlearning):')
print(base.to_string(index=False))
if 'retain_only' in adapters:
    print('Retain-only reference target and downstream reconstruction:')
    print(summary[(summary.checkpoint=='retain_only')&(summary.split=='forget')].to_string(index=False))
print('Important: MQuAKE-T has real temporal knowledge; original answers may be outdated as of run date.')
print('Important: direct and 2-hop correctness are scored against DIFFERENT gold entities.')
print('Important: inspect machine translations and raw model predictions manually.')
print('Saved CSVs:',*[p.name for p in ROOT.glob('*.csv')],sep='\n- ')

**No claimed publication result yet:** use 3+ training seeds, stronger alias/semantic adjudication, manual translation audit, real-data temporal ambiguity review, matched-bootstrap CIs, and a retention-utility evaluation. Repeating this with MLaKE or a separate published multilingual multi-hop benchmark would strengthen external validity.